In [1]:
# CONFIGURATION
import numpy as np
import random
import os

from dotenv import load_dotenv
load_dotenv()

# Paths
TRAIN_PATH = "data/training_scaled.csv"
VAL_PATH = "data/val_scaled.csv"
TEST_PATH = "data/test_scaled.csv"
SCALER_PATH = "artifacts/minmax_scaler.pkl"
PREPROC_PATH = "data/nsrdb_preprocessed.csv"

# Daily aggregate paths (medium/long horizon models)
# Daily scaled splits produced by preprocessing
DAILY_TRAIN_PATH = "data/nsrdb_daily_train.csv"
DAILY_VAL_PATH = "data/nsrdb_daily_val.csv"
DAILY_TEST_PATH = "data/nsrdb_daily_test.csv"
DAILY_TRAIN_SCALED_PATH = "data/daily_train_scaled.csv"
DAILY_VAL_SCALED_PATH = "data/daily_val_scaled.csv"
DAILY_TEST_SCALED_PATH = "data/daily_test_scaled.csv"

# Target
TARGET_COL = "ghi"

# Sequence config
WINDOW_SIZE = 24
SHORT_HORIZONS = [6,12] # Hourly
MEDIUM_HORIZONS = [7, 14] # Daily
LONG_HORIZONS = [28, 56, 84, 168, 336] # Daily
DAILY_WINDOW = 30

# LSTM architecture
LSTM_UNITS_1 = 128
LSTM_UNITS_2 = 64
DROPOUT_RATE = .2
DENSE_UNITS = 32

# XGBoost
XGB_N_ESTIMATORS = 500
XGB_LEARNING_RATE = .05
XGB_MAX_DEPTH = 6
XGB_EARLY_STOPPING = 20

# Prophet
PROPHET_REGRESSORS = ["temperature", "relative_humidity", "wind_speed", "solar_zenith_angle" , "surface_albedo"]

# Training
EPOCHS = 100
BATCH_SIZE = 64
PATIENCE = 10
LEARNING_RATE = 0.001

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

os.makedirs("artifacts", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

In [2]:
# IMPORTS
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
tf.random.set_seed(RANDOM_SEED)

from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, Input
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.optimizers import Adam

import xgboost as xgb
from prophet import Prophet

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

I0000 00:00:1780275428.861908  252717 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


/home/elkk/anaconda3/envs/Solar/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# LOAD DATA
# Hourly scaled splits for LSTM and XGBoost sequence building
train_scaled = pd.read_csv(TRAIN_PATH, index_col="datetime", parse_dates=True)
val_scaled = pd.read_csv(VAL_PATH, index_col="datetime", parse_dates=True)
test_scaled = pd.read_csv(TEST_PATH, index_col="datetime", parse_dates=True)

# Unscaled data for Prophet
df_full = pd.read_csv(PREPROC_PATH, index_col="datetime", parse_dates=True)

# Fitted hourly scaler for inverse transforming predictions back to W/m^2
scaler = joblib.load(SCALER_PATH)

feature_cols = train_scaled.columns.tolist()
target_idx = feature_cols.index(TARGET_COL)

# Reconstruct unscaled splits using the same 70/15/15 splits
n = len(df_full)
df_train_raw = df_full.iloc[:int(n*.7)]
df_val_raw = df_full.iloc[int(n*.7):int(n*.85)]
df_test_raw = df_full.iloc[int(n*.85):]

print(f"Train scaled: {train_scaled.shape}")
print(f"Val scaled: {val_scaled.shape}")
print(f"Test scaled: {test_scaled.shape}")
print(f"Target: '{TARGET_COL}' at index {target_idx}")

# Daily unscaled splits (used by Prophet long horizons)
daily_train = pd.read_csv(DAILY_TRAIN_PATH, index_col="datetime", parse_dates=True)
daily_val = pd.read_csv(DAILY_VAL_PATH, index_col="datetime", parse_dates=True)
daily_test = pd.read_csv(DAILY_TEST_PATH, index_col="datetime", parse_dates=True)

# Daily scaled splits produced by preprocessing (used by XGBoost daily)
daily_train_scaled = pd.read_csv(DAILY_TRAIN_SCALED_PATH, index_col="datetime", parse_dates=True)
daily_val_scaled = pd.read_csv(DAILY_VAL_SCALED_PATH, index_col="datetime", parse_dates=True)
daily_test_scaled = pd.read_csv(DAILY_TEST_SCALED_PATH, index_col="datetime", parse_dates=True)

daily_feature_cols = daily_train_scaled.columns.tolist()
daily_target_idx = daily_feature_cols.index(TARGET_COL)

print(f"Daily train: {daily_train_scaled.shape}")
print(f"Daily val: {daily_val_scaled.shape}")
print(f"Daily test: {daily_test_scaled.shape}")
print(f"Daily target '{TARGET_COL}' at index {daily_target_idx}")

Train scaled: (61370, 17)
Val scaled: (13151, 17)
Test scaled: (13151, 17)
Target: 'ghi' at index 0
Daily train: (2557, 15)
Daily val: (548, 15)
Daily test: (548, 15)
Daily target 'ghi' at index 0


In [4]:
# SHARED SEQUENCE BUILDER
# Sliding window builder shared by LSTM and XGBoost
# Returns X: (samples, window, features) and y_dict: {horizon: 1D array}
def build_sequences(data: np.ndarray, window: int, horizons: list, tgt_idx) -> tuple:
    X, ys = [], {h: [] for h in horizons}
    max_h = max(horizons)
    for i in range(window, len(data) - max_h +1):
        X.append(data[i-window : i, :])
        for h in horizons:
            ys[h].append(data[i + h - 1, tgt_idx])
    return np.array(X), {h: np.array(v) for h, v in ys.items()}

train_arr = train_scaled.values
val_arr = val_scaled.values
test_arr = test_scaled.values

X_train, y_train = build_sequences(train_arr, WINDOW_SIZE, SHORT_HORIZONS, target_idx)
X_val, y_val = build_sequences(val_arr, WINDOW_SIZE, SHORT_HORIZONS, target_idx)
X_test, y_test = build_sequences(test_arr, WINDOW_SIZE, SHORT_HORIZONS, target_idx)

print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")

X_train: (61335, 24, 17)
X_val: (13116, 24, 17)
X_test: (13116, 24, 17)


In [5]:
# BUILD DAILY SEQUENCES FOR MEDIUM HORIZON XGBOOST
daily_train_arr = daily_train_scaled.values
daily_val_arr = daily_val_scaled.values
daily_test_arr = daily_test_scaled.values

X_daily_train, y_daily_train = build_sequences(daily_train_arr, DAILY_WINDOW, MEDIUM_HORIZONS, daily_target_idx)
X_daily_val, y_daily_val = build_sequences(daily_val_arr, DAILY_WINDOW, MEDIUM_HORIZONS, daily_target_idx)
X_daily_test, y_daily_test = build_sequences(daily_test_arr, DAILY_WINDOW, MEDIUM_HORIZONS, daily_target_idx)

print(f"X_daily_train: {X_daily_train.shape}")
print(f"X_daily_val: {X_daily_val.shape}")
print(f"X_daily_test: {X_daily_test.shape}")
print(f"Targets: {list(y_daily_train.keys())} (days ahead)")

X_daily_train: (2514, 30, 15)
X_daily_val: (505, 30, 15)
X_daily_test: (505, 30, 15)
Targets: [7, 14] (days ahead)


In [6]:
# BUILD LSTM
def build_lstm(window: int, n_features: int, n_outputs: int) -> tf.keras.Model:
    model = Sequential([
        Input(shape=(window, n_features)),
        LSTM(LSTM_UNITS_1, return_sequences=True, use_cudnn=False),
        Dropout(DROPOUT_RATE),
        LSTM(LSTM_UNITS_2, return_sequences=False, use_cudnn=False),
        Dropout(DROPOUT_RATE),
        Dense(DENSE_UNITS, activation="relu"),
        Dense(n_outputs)
    ])
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="mse",
        metrics=["mae"]
    )
    return model

lstm_model = build_lstm(window=WINDOW_SIZE, n_features=X_train.shape[2], n_outputs=len(SHORT_HORIZONS))
lstm_model.summary()
print("\nLSTM model built - not yet trained")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 128)        │        74,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 126,306 (493.38 KB)

 Trainable params: 126,306 (493.38 KB)

 Non-trainable params: 0 (0.00 B)


LSTM model built - not yet trained


In [7]:
# BUILD SHORT HORIZON XGBOOST MODELS
# One XGBRegressor per horizon, built here, trained later
xgb_models = {}
for h in SHORT_HORIZONS:
    xgb_models[h] = xgb.XGBRegressor(
        n_estimators=XGB_N_ESTIMATORS,
        learning_rate=XGB_LEARNING_RATE,
        max_depth=XGB_MAX_DEPTH,
        subsample=.8,
        colsample_bytree=.8,
        random_state=RANDOM_SEED,
        device="cuda",
        n_jobs=1,
        early_stopping_rounds=XGB_EARLY_STOPPING,
        eval_metric="rmse",
    )
    print(f"XGBoost h={h}h model built - not yet trained")


XGBoost h=6h model built - not yet trained
XGBoost h=12h model built - not yet trained


In [8]:
# BUILD MEDIUM HORIZON XGBOOST MODELS
# One XGBRegressor per horizon, built here, trained later
xgb_daily_models = {}
for h in MEDIUM_HORIZONS:
    xgb_daily_models[h] = xgb.XGBRegressor(
        n_estimators=XGB_N_ESTIMATORS,
        learning_rate=XGB_LEARNING_RATE,
        max_depth=XGB_MAX_DEPTH,
        subsample=.8,
        colsample_bytree=.8,
        random_state=RANDOM_SEED,
        device="cuda",
        n_jobs=1,
        early_stopping_rounds=XGB_EARLY_STOPPING,
        eval_metric="rmse",
    )
    print(f"XGBoost daily h={h}h model built - not yet trained")

XGBoost daily h=7h model built - not yet trained
XGBoost daily h=14h model built - not yet trained


In [9]:
# BUILD SHORT HORIZON PROPHET MODELS
# One model per horizon
# Target shifted forward h hours before fitting
prophet_models = {}
for h in SHORT_HORIZONS:
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=True,
        seasonality_mode="multiplicative",
        changepoint_prior_scale=.05,
    )
    for col in PROPHET_REGRESSORS:
        m.add_regressor(col)
    prophet_models[h] = m
    print(f"Prophet h={h}h model built - not yet trained")

Prophet h=6h model built - not yet trained
Prophet h=12h model built - not yet trained


In [10]:
# BUILD LONG HORIZON PROPHET MODELS
# One model per horizon
# Target shifted forward h hours before fitting
prophet_long_models = {}
for h in LONG_HORIZONS:
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        seasonality_mode="multiplicative",
        changepoint_prior_scale=.05,
        seasonality_prior_scale=10.0,
    )
    for col in PROPHET_REGRESSORS:
        m.add_regressor(col)
    prophet_long_models[h] = m
    print(f"Prophet long h={h}d model built - not yet trained")

Prophet long h=28d model built - not yet trained
Prophet long h=56d model built - not yet trained
Prophet long h=84d model built - not yet trained
Prophet long h=168d model built - not yet trained
Prophet long h=336d model built - not yet trained


In [11]:
# TRAIN LSTM
# Stack horizon targets into a single (sampels, n_outputs) matrix
y_train_mat = np.column_stack([y_train[h] for h in SHORT_HORIZONS])
y_val_mat = np.column_stack([y_val[h] for h in SHORT_HORIZONS])

callbacks = [
    EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath="artifacts/lstm_best.keras", monitor="val_loss", save_best_only=True, verbose=0)
]

history = lstm_model.fit(
    X_train,
    y_train_mat,
    validation_data=(X_val,y_val_mat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print(f"\nStopped at epoch: {len(history.history['loss'])}")
print(f"Best val_loss: {min(history.history['val_loss']):.6f}")

Epoch 1/100


  1/959 ━━━━━━━━━━━━━━━━━━━━ 37:17 2s/step - loss: 0.1175 - mae: 0.2316

 12/959 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0692 - mae: 0.1900  

 23/959 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0585 - mae: 0.1760

 34/959 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0517 - mae: 0.1647

...

950/959 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0069 - mae: 0.0434

959/959 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.0068 - mae: 0.0433 - val_loss: 0.0122 - val_mae: 0.0537


Epoch 24: early stopping


Restoring model weights from the end of the best epoch: 14.



Stopped at epoch: 24
Best val_loss: 0.010616


In [12]:
# TRAIN SHORT HORIZON XGBOOST
# Flatten 3D sequences to 2D for XGBoost: (samples, window * features)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

for h in SHORT_HORIZONS:
    xgb_models[h].fit(X_train_flat, y_train[h], eval_set=[(X_val_flat, y_val[h])], verbose=False)
    print(f"XGBoost h={h}h trained - best iteration: {xgb_models[h].best_iteration}")

joblib.dump(xgb_models, "artifacts/xgb_models.pkl")
print("\nXGBoost models saved to artifacts/xgb_models.pkl")

sh: line 1: nvidia-smi: command not found


XGBoost h=6h trained - best iteration: 81


XGBoost h=12h trained - best iteration: 80

XGBoost models saved to artifacts/xgb_models.pkl


In [13]:
# TRAIN MEDIUM HORIZON XGBOOST
X_daily_train_flat = X_daily_train.reshape(X_daily_train.shape[0], -1)
X_daily_val_flat = X_daily_val.reshape(X_daily_val.shape[0], -1)
X_daily_test_flat = X_daily_test.reshape(X_daily_test.shape[0], -1)

for h in MEDIUM_HORIZONS:
    xgb_daily_models[h].fit(X_daily_train_flat, y_daily_train[h], eval_set=[(X_daily_val_flat, y_daily_val[h])], verbose=False)
    print(f"XGBoost daily h={h}h trained - best iteration: {xgb_daily_models[h].best_iteration}")

joblib.dump(xgb_daily_models, "artifacts/xgb_daily_models.pkl")
print("\nXGBoost medium models saved to artifacts/xgb_daily_models.pkl")

XGBoost daily h=7h trained - best iteration: 55


XGBoost daily h=14h trained - best iteration: 68

XGBoost medium models saved to artifacts/xgb_daily_models.pkl


In [14]:
# TRAIN SHORT HORIZON PROPHET
# Shift GHI target forward by h hours so Prophet predicts h steps ahead
for h in SHORT_HORIZONS:
    train_prophet = df_train_raw[[TARGET_COL] + PROPHET_REGRESSORS].copy()
    train_prophet["y"] = train_prophet[TARGET_COL].shift(-h)
    train_prophet["ds"] = train_prophet.index
    train_prophet = train_prophet.dropna()[["ds", "y"] + PROPHET_REGRESSORS]

    prophet_models[h].fit(train_prophet)
    print(f"Prophet h={h}h trained on {len(train_prophet):,} rows")

19:59:00 - cmdstanpy - INFO - Chain [1] start processing


19:59:38 - cmdstanpy - INFO - Chain [1] done processing


Prophet h=6h trained on 61,364 rows


19:59:40 - cmdstanpy - INFO - Chain [1] start processing


20:00:37 - cmdstanpy - INFO - Chain [1] done processing


Prophet h=12h trained on 61,358 rows


In [15]:
# TRAIN LONG HORIZON PROPHET
# Uses the unscaled daily GHI series with regressors
# Target shifted h days forward
df_daily_train_raw = daily_train.copy()

for h in LONG_HORIZONS:
    cols = [TARGET_COL] + PROPHET_REGRESSORS
    prophet_df = df_daily_train_raw[cols].copy()
    prophet_df["y"] = prophet_df[TARGET_COL].shift(-h)
    prophet_df["ds"] = prophet_df.index
    prophet_df = prophet_df.dropna()[["ds", "y"] + PROPHET_REGRESSORS]

    # Clip any negative GHI values (physically impossible/rare edge at day boundaries)
    prophet_df["y"] = prophet_df["y"].clip(lower=0)

    prophet_long_models[h].fit(prophet_df)

    weeks = h // 7

    print(f"Prophet long h={h}d ({weeks}w)  train on {len(prophet_df):,} days")

# Save all long horizon Prophet models
for h in LONG_HORIZONS:
    joblib.dump(prophet_long_models[h], f"artifacts/prophet_long_h{h}d.pkl")
    print(f"Saved to artifacts/prophet_long_h{h}d.pkl")

print("\nAll long horizon Prophet models saved.")

20:00:37 - cmdstanpy - INFO - Chain [1] start processing


20:00:37 - cmdstanpy - INFO - Chain [1] done processing


20:00:37 - cmdstanpy - INFO - Chain [1] start processing


Prophet long h=28d (4w)  train on 2,529 days


20:00:37 - cmdstanpy - INFO - Chain [1] done processing


...

20:00:37 - cmdstanpy - INFO - Chain [1] done processing


20:00:37 - cmdstanpy - INFO - Chain [1] start processing


Prophet long h=168d (24w)  train on 2,389 days


20:00:38 - cmdstanpy - INFO - Chain [1] done processing


Prophet long h=336d (48w)  train on 2,221 days
Saved to artifacts/prophet_long_h28d.pkl
Saved to artifacts/prophet_long_h56d.pkl
Saved to artifacts/prophet_long_h84d.pkl
Saved to artifacts/prophet_long_h168d.pkl
Saved to artifacts/prophet_long_h336d.pkl

All long horizon Prophet models saved.


In [16]:
# SAVE REMAINING ARTIFACTS
lstm_model.save("artifacts/lstm_final.keras")
print("LSTM saved to artifacts/lstm_final.keras")

# XGBoost already saved above
print("XGBoost saved to artifacts/xgb_models.pkl")

# Prophet short horizon models
for h in SHORT_HORIZONS:
    joblib.dump(prophet_models[h], f"artifacts/prophet_h{h}.pkl")
    print(f"Prophet h={h}h saved to artifacts/prophet_h{h}.pkl")

print("\nAll artifacts:")
for f in sorted(os.listdir("artifacts")):
    size = os.path.getsize(f"artifacts/{f}") / 1024
    print(f"artifacts/{f} ({size:.1f} KB)")

LSTM saved to artifacts/lstm_final.keras
XGBoost saved to artifacts/xgb_models.pkl


Prophet h=6h saved to artifacts/prophet_h6.pkl
Prophet h=12h saved to artifacts/prophet_h12.pkl

All artifacts:
artifacts/best_hyperparams.pkl (0.6 KB)
artifacts/daily_minmax_scaler.pkl (1.7 KB)
artifacts/lstm_best.keras (1521.1 KB)
artifacts/lstm_final.keras (1521.1 KB)
artifacts/lstm_tuned.keras (458.7 KB)
artifacts/minmax_scaler.pkl (1.8 KB)
artifacts/prophet_h12.pkl (7842.9 KB)
artifacts/prophet_h6.pkl (7843.7 KB)
artifacts/prophet_long_h168d.pkl (312.4 KB)
artifacts/prophet_long_h28d.pkl (330.0 KB)
artifacts/prophet_long_h336d.pkl (290.8 KB)
artifacts/prophet_long_h56d.pkl (326.5 KB)
artifacts/prophet_long_h84d.pkl (323.0 KB)
artifacts/prophet_long_tuned_h168d.pkl (312.4 KB)
artifacts/prophet_long_tuned_h28d.pkl (330.0 KB)
artifacts/prophet_long_tuned_h336d.pkl (290.8 KB)
artifacts/prophet_long_tuned_h56d.pkl (326.5 KB)
artifacts/prophet_long_tuned_h84d.pkl (323.0 KB)
artifacts/prophet_tuned_h12.pkl (7842.9 KB)
artifacts/prophet_tuned_h6.pkl (7843.7 KB)
artifacts/xgb_daily_models.